In [1]:
import os

import argparse
import numpy as np
from tqdm import tqdm

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import transforms
from accelerate import Accelerator


import pandas as pd

from generative_models.sgm.modules.encoders.modules import FrozenOpenCLIPImageEmbedder
from models import GNet8_Encoder

# tf32 data type is faster than standard float32
torch.backends.cuda.matmul.allow_tf32 = True

# custom functions #
import utils

accelerator = Accelerator(split_batches=False, mixed_precision="fp16")
device = accelerator.device
print("device:",device)

/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


device: cuda


In [2]:
from itertools import product
delays = [0, 1, 3, 5, 10, 15, 20]
seeds = [0, 1, 2, 3, 4, 5]
num_avg = [1, 3]

grid = list(product(delays, seeds, num_avg))

In [3]:
# if running this interactively, can specify jupyter_args here for argparser to use
if utils.is_interactive():
    model_name = "final_subj01_pretrained_multisubject_subj01_1024hid_nolow_300ep_1sess_cindy"
    
    subj = 1
    
    data_path = "mindeyev2_data"
    cache_dir = "mindeyev2_data"
    seed = 4
    num_avg = 3
    print("model_name:", model_name)

    jupyter_args = f"--model_name={model_name} --subj={subj} --data_path={data_path} --cache_dir={cache_dir}"
    jupyter_args += " --subset_path=./special515_indices.csv"
    # jupyter_args += f" --seed={seed} --eval_dir=./evals/{model_name}_avg_{num_avg}/{seed}/"
    jupyter_args += f" --grid_idx=0"
    print(jupyter_args)
    jupyter_args = jupyter_args.split()
    
    from IPython.display import clear_output # function to clear print outputs in cell
    %load_ext autoreload 
    # this allows you to change functions in models.py or utils.py and have this notebook automatically update with your revisions
    %autoreload 2 

model_name: final_subj01_pretrained_multisubject_subj01_1024hid_nolow_300ep_1sess_cindy
--model_name=final_subj01_pretrained_multisubject_subj01_1024hid_nolow_300ep_1sess_cindy --subj=1 --data_path=mindeyev2_data --cache_dir=mindeyev2_data --subset_path=./special515_indices.csv --grid_idx=0


In [ ]:
parser = argparse.ArgumentParser(description="Model Training Configuration")
parser.add_argument(
    "--model_name", type=str, default="testing",
    help="name of model, used for ckpt saving and wandb logging (if enabled)",
)
parser.add_argument(
    "--data_path", type=str, default=os.getcwd(),
    help="Path to where NSD data is stored / where to download it to",
)
parser.add_argument(
    "--cache_dir", type=str, default=os.getcwd(),
    help="Path to where misc. files downloaded from huggingface are stored. Defaults to current src directory.",
)
parser.add_argument(
    "--subj",type=int, default=1, choices=[1,2,3,4,5,6,7,8],
    help="Evaluate on which subject?",
)
parser.add_argument(
    "--subset_path", default=None
)
# parser.add_argument(
#     "--eval_dir", default=None, help="path to dump metrics"
# )
parser.add_argument(
    "--blurry_recon", action="store_true"
)

parser.add_argument(
    "--grid_idx", type=int, required=True
)

if utils.is_interactive():
    args = parser.parse_args(jupyter_args)
else:
    args = parser.parse_args()

# create global variables without the args prefix
for attribute_name in vars(args).keys():
    globals()[attribute_name] = getattr(args, attribute_name)

delay, seed, num_avg = grid[grid_idx]
print(f"{delay=}, {seed=}, {num_avg=}")

base_dir = "/scratch/am10150/projects/MindEyeV2/src/evals/"
eval_dir = f"rtnorm_subj1_delay={delay}/{model_name}_avg_{num_avg}/{seed}"
eval_dir = os.path.join(base_dir, eval_dir)
print(f"{eval_dir=}")

# seed all random functions
utils.seed_everything(seed)

delay=0, seed=0, num_avg=1
eval_dir='/scratch/am10150/projects/MindEyeV2/src/evals/rtnorm_subj1_delay=0/final_subj01_pretrained_multisubject_subj01_1024hid_nolow_300ep_1sess_cindy_avg_1/0'


In [5]:
# load indices if exists
indices = None
if args.subset_path:
    indices = pd.read_csv(args.subset_path)['indices'].tolist()

In [6]:
indices

[828,
 362,
 557,
 391,
 699,
 822,
 717,
 254,
 782,
 356,
 253,
 659,
 282,
 486,
 12,
 81,
 406,
 69,
 295,
 74,
 724,
 385,
 988,
 464,
 276,
 986,
 855,
 732,
 52,
 412,
 929,
 485,
 605,
 336,
 931,
 569,
 968,
 762,
 531,
 193,
 294,
 185,
 377,
 45,
 35,
 314,
 365,
 424,
 144,
 849]

# Evals

In [36]:
# Load ground truths, you can find these files on huggingface: https://huggingface.co/datasets/pscotti/mindeyev2/tree/main/evals
all_images = torch.load(f"mindeyev2_data/evals/all_images_special515.pt")
all_captions = torch.load(f"mindeyev2_data/evals/all_captions.pt")

In [37]:
all_recons_path = os.path.join(eval_dir, f"{model_name}_all_recons.pt")
print("all_recons_path:", all_recons_path)
all_recons = torch.load(all_recons_path)

# Residual submodule
all_clipvoxels = torch.load(os.path.join(eval_dir, f"{model_name}_all_clipvoxels.pt"))
# Low-level submodule
if blurry_recon:
    all_blurryrecons = torch.load(os.path.join(eval_dir, f"{model_name}_all_blurryrecons.pt"))
# GIT predicted captions
all_predcaptions = torch.load(os.path.join(eval_dir, f"{model_name}_all_predcaptions.pt"))

# model name
model_name_plus_suffix = f"{model_name}_all_recons"
print(model_name_plus_suffix)
print(all_images.shape, all_recons.shape)

all_recons_path: /scratch/am10150/projects/MindEyeV2/src/evals/rtnorm_subj1_delay=0/final_subj01_pretrained_multisubject_subj01_1024hid_nolow_300ep_1sess_cindy_avg_1/0/final_subj01_pretrained_multisubject_subj01_1024hid_nolow_300ep_1sess_cindy_all_recons.pt
final_subj01_pretrained_multisubject_subj01_1024hid_nolow_300ep_1sess_cindy_all_recons
torch.Size([50, 3, 224, 224]) torch.Size([50, 3, 256, 256])


In [38]:
# # Create full grid of recon comparisons 
# from PIL import Image

# imsize = 150
# if all_images.shape[-1] != imsize:
#     all_images = transforms.Resize((imsize,imsize))(all_images).float()
# if all_recons.shape[-1] != imsize:
#     all_recons = transforms.Resize((imsize,imsize))(all_recons).float()

# num_images = all_recons.shape[0]
# num_rows = (2 * num_images + 9) // 10

# # Interleave tensors
# merged = torch.stack([val for pair in zip(all_images, all_recons) for val in pair], dim=0)

# # Calculate grid size
# grid = torch.zeros((num_rows * 10, 3, all_recons.shape[-1], all_recons.shape[-1]))

# # Populate the grid
# grid[:2*num_images] = merged
# grid_images = [transforms.functional.to_pil_image(grid[i]) for i in range(num_rows * 10)]

# # Create the grid image
# grid_image = Image.new('RGB', (all_recons.shape[-1]*10, all_recons.shape[-1] * num_rows))  # 10 images wide

# # Paste images into the grid
# for i, img in enumerate(grid_images):
#     grid_image.paste(img, (all_recons.shape[-1] * (i % 10), all_recons.shape[-1] * (i // 10)))

# grid_image.save(f"{model_name_plus_suffix[:-3]}_1000recons.png")

In [39]:
imsize = 256
if all_images.shape[-1] != imsize:
    all_images = transforms.Resize((imsize,imsize))(all_images).float()
if all_recons.shape[-1] != imsize:
    all_recons = transforms.Resize((imsize,imsize))(all_recons).float()

if blurry_recon:
    if all_blurryrecons.shape[-1] != imsize:
        all_blurryrecons = transforms.Resize((imsize,imsize))(all_blurryrecons).float()
    
if "enhanced" in model_name_plus_suffix:
    print("weighted averaging to improve low-level evals")
    all_recons = all_recons*.75 + all_blurryrecons*.25

/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


In [43]:
if indices:
    # all_images = all_images[indices]
    # all_captions = all_captions[indices]

    # when we have done recons only for 50 images then we have reorder indices based on indices and choose them
    indices2idx_ord = {ind: idx for idx, ind in enumerate(sorted(indices))}
    idx2choose = [indices2idx_ord[ind] for ind in indices]
    
    all_recons = all_recons[idx2choose]
    if blurry_recon:
        all_blurryrecons = all_blurryrecons[idx2choose]
    all_clipvoxels = all_clipvoxels[idx2choose]


In [44]:
print(all_recons.shape, all_images.shape, all_captions.shape, all_clipvoxels.shape)

torch.Size([50, 3, 256, 256]) torch.Size([50, 3, 256, 256]) (1000,) torch.Size([50, 256, 1664])


# Retrieval eval

In [41]:
# Load embedding model
clip_img_embedder = FrozenOpenCLIPImageEmbedder(
    arch="ViT-bigG-14",
    version="laion2b_s39b_b160k",
    output_tokens=True,
    only_tokens=True,
)
clip_img_embedder.to(device)

clip_seq_dim = 256
clip_emb_dim = 1664

In [45]:
from scipy import stats
percent_correct_fwds, percent_correct_bwds = [], []
percent_correct_fwd, percent_correct_bwd = None, None

n_runs = 1
with torch.cuda.amp.autocast(dtype=torch.float16):
    for test_i, loop in enumerate(tqdm(range(n_runs))):
        # random_samps = np.random.choice(np.arange(len(all_images)), size=sample_size, replace=False)
        emb = clip_img_embedder(all_images.to(device)).float() # CLIP-Image
        emb_ = all_clipvoxels.to(device).float() # CLIP-Brain

        # flatten if necessary
        emb = emb.reshape(len(emb),-1)
        emb_ = emb_.reshape(len(emb_),-1)

        # l2norm 
        emb = nn.functional.normalize(emb,dim=-1)
        emb_ = nn.functional.normalize(emb_,dim=-1)

        labels = torch.arange(len(emb)).to(device)
        bwd_sim = utils.batchwise_cosine_similarity(emb,emb_)  # clip, brain
        fwd_sim = utils.batchwise_cosine_similarity(emb_,emb)  # brain, clip

        # assert len(bwd_sim) == sample_size
        percent_correct_fwds = np.append(percent_correct_fwds, utils.topk(fwd_sim, labels, k=1).item())
        percent_correct_bwds = np.append(percent_correct_bwds, utils.topk(bwd_sim, labels, k=1).item())

        if test_i==0:
            print("Loop 0:",percent_correct_fwds, percent_correct_bwds)
            
percent_correct_fwd = np.mean(percent_correct_fwds)
fwd_sd = np.std(percent_correct_fwds) / np.sqrt(len(percent_correct_fwds))
fwd_ci = stats.norm.interval(0.95, loc=percent_correct_fwd, scale=fwd_sd)

percent_correct_bwd = np.mean(percent_correct_bwds)
bwd_sd = np.std(percent_correct_bwds) / np.sqrt(len(percent_correct_bwds))
bwd_ci = stats.norm.interval(0.95, loc=percent_correct_bwd, scale=bwd_sd)

print(f"fwd percent_correct: {percent_correct_fwd:.4f} 95% CI: [{fwd_ci[0]:.4f},{fwd_ci[1]:.4f}]")
print(f"bwd percent_correct: {percent_correct_bwd:.4f} 95% CI: [{bwd_ci[0]:.4f},{bwd_ci[1]:.4f}]")

fwd_sim = np.array(fwd_sim.cpu())
bwd_sim = np.array(bwd_sim.cpu())

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.50it/s]

Loop 0: [0.44] [0.34]
fwd percent_correct: 0.4400 95% CI: [nan,nan]
bwd percent_correct: 0.3400 95% CI: [nan,nan]


## 2-way identification

In [46]:
from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names

@torch.no_grad()
def two_way_identification(all_recons, all_images, model, preprocess, feature_layer=None, return_avg=True):
    preds = model(torch.stack([preprocess(recon) for recon in all_recons], dim=0).to(device))
    reals = model(torch.stack([preprocess(indiv) for indiv in all_images], dim=0).to(device))
    if feature_layer is None:
        preds = preds.float().flatten(1).cpu().numpy()
        reals = reals.float().flatten(1).cpu().numpy()
    else:
        preds = preds[feature_layer].float().flatten(1).cpu().numpy()
        reals = reals[feature_layer].float().flatten(1).cpu().numpy()

    r = np.corrcoef(reals, preds)
    r = r[:len(all_images), len(all_images):]
    congruents = np.diag(r)

    success = r < congruents
    success_cnt = np.sum(success, 0)

    if return_avg:
        perf = np.mean(success_cnt) / (len(all_images)-1)
        return perf
    else:
        return success_cnt, len(all_images)-1

## PixCorr

In [47]:
preprocess = transforms.Compose([
    transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR),
])

# Flatten images while keeping the batch dimension
all_images_flattened = preprocess(all_images).reshape(len(all_images), -1).cpu()
all_recons_flattened = preprocess(all_recons).view(len(all_recons), -1).cpu()

print(all_images_flattened.shape)
print(all_recons_flattened.shape)

corrsum = 0
for i in tqdm(range(len(all_images))):
    corrsum += np.corrcoef(all_images_flattened[i], all_recons_flattened[i])[0][1]
corrmean = corrsum / len(all_images)

pixcorr = corrmean
print(pixcorr)

torch.Size([50, 541875])
torch.Size([50, 541875])


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 165.32it/s]

0.14499646296244018


## SSIM

In [48]:
# see https://github.com/zijin-gu/meshconv-decoding/issues/3
from skimage.color import rgb2gray
from skimage.metrics import structural_similarity as ssim

preprocess = transforms.Compose([
    transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR), 
])

# convert image to grayscale with rgb2grey
img_gray = rgb2gray(preprocess(all_images).permute((0,2,3,1)).cpu())
recon_gray = rgb2gray(preprocess(all_recons).permute((0,2,3,1)).cpu())
print("converted, now calculating ssim...")

ssim_score=[]
for im,rec in tqdm(zip(img_gray,recon_gray),total=len(all_images)):
    ssim_score.append(ssim(rec, im, multichannel=True, gaussian_weights=True, sigma=1.5, use_sample_covariance=False, data_range=1.0))

ssim = np.mean(ssim_score)
print(ssim)

converted, now calculating ssim...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 63.07it/s]

0.3190762001486249


## AlexNet

In [49]:
from torchvision.models import alexnet, AlexNet_Weights
alex_weights = AlexNet_Weights.IMAGENET1K_V1

alex_model = create_feature_extractor(alexnet(weights=alex_weights), return_nodes=['features.4','features.11']).to(device)
alex_model.eval().requires_grad_(False)

# see alex_weights.transforms()
preprocess = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

layer = 'early, AlexNet(2)'
print(f"\n---{layer}---")
all_per_correct = two_way_identification(all_recons.to(device).float(), all_images, 
                                                          alex_model, preprocess, 'features.4')
alexnet2 = np.mean(all_per_correct)
print(f"2-way Percent Correct: {alexnet2:.4f}")

layer = 'mid, AlexNet(5)'
print(f"\n---{layer}---")
all_per_correct = two_way_identification(all_recons.to(device).float(), all_images, 
                                                          alex_model, preprocess, 'features.11')
alexnet5 = np.mean(all_per_correct)
print(f"2-way Percent Correct: {alexnet5:.4f}")


---early, AlexNet(2)---
2-way Percent Correct: 0.7192

---mid, AlexNet(5)---
2-way Percent Correct: 0.7727


## InceptionV3

In [31]:
from torchvision.models import inception_v3, Inception_V3_Weights
weights = Inception_V3_Weights.DEFAULT
inception_model = create_feature_extractor(inception_v3(weights=weights), 
                                           return_nodes=['avgpool']).to(device)
inception_model.eval().requires_grad_(False)

# see weights.transforms()
preprocess = transforms.Compose([
    transforms.Resize(342, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

all_per_correct = two_way_identification(all_recons, all_images,
                                        inception_model, preprocess, 'avgpool')
        
inception = np.mean(all_per_correct)
print(f"2-way Percent Correct: {inception:.4f}")

/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/torchvision/models/feature_extraction.py:174: UserWarning: NOTE: The nodes obtained by tracing the model in eval mode are a subsequence of those obtained in train mode. When choosing nodes for feature extraction, you may need to specify output nodes for train and eval mode separately.
  warnings.warn(msg + suggestion_msg)


2-way Percent Correct: 0.6608


## CLIP

In [32]:
import clip
clip_model, preprocess = clip.load("ViT-L/14", device=device)

preprocess = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                         std=[0.26862954, 0.26130258, 0.27577711]),
])

all_per_correct = two_way_identification(all_recons, all_images,
                                        clip_model.encode_image, preprocess, None) # final layer
clip_ = np.mean(all_per_correct)
print(f"2-way Percent Correct: {clip_:.4f}")

2-way Percent Correct: 0.6282


## Efficient Net

In [33]:
import scipy as sp
from torchvision.models import efficientnet_b1, EfficientNet_B1_Weights
weights = EfficientNet_B1_Weights.DEFAULT
eff_model = create_feature_extractor(efficientnet_b1(weights=weights), 
                                    return_nodes=['avgpool'])
eff_model.eval().requires_grad_(False)

# see weights.transforms()
preprocess = transforms.Compose([
    transforms.Resize(255, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

gt = eff_model(preprocess(all_images))['avgpool']
gt = gt.reshape(len(gt),-1).cpu().numpy()
fake = eff_model(preprocess(all_recons))['avgpool']
fake = fake.reshape(len(fake),-1).cpu().numpy()

effnet = np.array([sp.spatial.distance.correlation(gt[i],fake[i]) for i in range(len(gt))]).mean()
print("Distance:",effnet)

Distance: 0.9128765236801918


## SwAV

In [34]:
swav_model = torch.hub.load('facebookresearch/swav:main', 'resnet50')
swav_model = create_feature_extractor(swav_model, 
                                    return_nodes=['avgpool'])
swav_model.eval().requires_grad_(False)

preprocess = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

gt = swav_model(preprocess(all_images))['avgpool']
gt = gt.reshape(len(gt),-1).cpu().numpy()
fake = swav_model(preprocess(all_recons))['avgpool']
fake = fake.reshape(len(fake),-1).cpu().numpy()

swav = np.array([sp.spatial.distance.correlation(gt[i], fake[i]) for i in range(len(gt))]).mean()
print("Distance:",swav)

Using cache found in /scratch/am10150/.cache/torch/hub/facebookresearch_swav_main
/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/scratch/am10150/projects/rtcloud-projects/mindeye/conf/.venv/lib64/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Distance: 0.5309929599493806


In [35]:
metric_names = ['pixcorr', 'ssim', 'alexnet2', 'alexnet5', 'inception', 'clip_', 'effnet', 'swav', 'fwd_acc', 'bwd_acc']
metric_values = [pixcorr, ssim, alexnet2, alexnet5, inception, clip_, effnet, swav, percent_correct_fwds[0], percent_correct_bwds[0]]

metrics_df = pd.DataFrame({'metric': metric_names, 'values': metric_values})
if eval_dir:
    save_path = os.path.join(eval_dir, "metrics.csv")
    print(f"saving in {save_path}")
    metrics_df.to_csv(save_path, index=False)

print(metrics_df)

saving in /scratch/am10150/projects/MindEyeV2/src/evals/rtnorm_subj1_delay=0/final_subj01_pretrained_multisubject_subj01_1024hid_nolow_300ep_1sess_cindy_avg_1/0/metrics.csv
      metric    values
0    pixcorr  0.144996
1       ssim  0.319076
2   alexnet2  0.719184
3   alexnet5  0.772653
4  inception  0.660816
5      clip_  0.628163
6     effnet  0.912877
7       swav  0.530993
8    fwd_acc  0.440000
9    bwd_acc  0.340000
